# Rich Club Coefficient

### Imports

In [2]:
from __future__ import annotations
import networkx as nx
import numpy as np
import pytest
from abc import ABC, abstractmethod

### Error Classes

In [3]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass

class EmptyGraphError(Exception):
    """Exception raised for empty graph. Nodes with no edges."""
    pass

def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

### Abstract Class

In [4]:
class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no density.")

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass

### Decorators

In [5]:
def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls

### Auxiliar Fxns

In [6]:
def remove_self_loops(G: nx.DiGraph, n_nodes):
    G.remove_edges_from([(i,i) for i in range(n_nodes)])
    return G

### Rich Club Coeffcient Class

The tendency of hubs to be well connected with each other is a phenomena called the rich-club phenomena. It can be measured by the rich club coefficient, where you analyze how connected nodes with a higher degree than k are, every k in the network is analyzed the same way.

The rich club coefficient can only be calculated for undirected networks and does not allow self-loops.[10.1109/LCOMM.2004.823426]

In [7]:
@return_distribution    
class Rich_Club(_Property):
    """Rich Club Coefficient.

    The Rich Club Coefficient for every degree in the graph is defined as the clustering coeffficient
    of nodes with a higher degree than the degree being evaluated.

    Methods:
        compute: Compute the rich club coefficient of the graph.
        norm_biol: NO IMPLEMENTATION.
        norm_network: NO IMPLEMENTATION.
    """
    __name__ = 'Rich Club Coefficient'

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)

    def compute(self) -> np.array:
        """Compute the Rich Club Coefficient.

        Returns:
            np.array: distribution of rich club coefficients, by degree in graph.
        """
        n_edges = self.G.number_of_edges()
        if n_edges == 0:
            raise EmptyGraphError("There are no edges. Can not form rich clubs with no edges.")
        
        self.A = nx.DiGraph
        self.A = remove_self_loops(self.G, self._n_nodes)
        dict_coeff = nx.rich_club_coefficient(self.A.to_undirected(), normalized=False)
        self._raw_value = np.fromiter(dict_coeff.values(), dtype=float)
        
        return self._raw_value

    @check_raw_value
    def norm_biol(self):
        raise NormalizationError("No biological normalization implemented.")

    @check_raw_value
    def norm_network(self):
        raise NormalizationError("No theoretical normalization implemented.")

### Testing

In [8]:
# Null graph
G = nx.DiGraph()
with pytest.raises(NullGraphError) as e_info:
    property = Rich_Club(G)

# Empty graph, allows instance from an empty graph, but does not compute
n_nodes= 6
G.add_nodes_from(range(n_nodes))
property = Rich_Club(G)
with pytest.raises(EmptyGraphError) as e_info:
    property.compute()

# add edges
# complete directed graph with self loops
G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = Rich_Club(G)

expected = np.array([1.,1.,1.,1.,1.])

assert np.array_equal(property.compute(), expected)
with pytest.raises(NormalizationError) as e_info:
    property.norm_network()
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol()

# add edges
# only half of the nodes are parents and regulate every node in the graph
G = nx.DiGraph()
n_nodes= 6
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = Rich_Club(G)

expected = np.array([0.8, 0.8, 0.8, 1. , 1. ])

assert np.allclose(property.compute(), expected)
with pytest.raises(NormalizationError) as e_info:
    property.norm_network()
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol()